# 02 · Segmentación de negocio — IVR Alkosto

Toma `df_pasos` y `df_traza` (salidas de `01_reconstruccion_traza.ipynb`) y construye las
variables de negocio para el tablero de Power BI y para filtrar el grafo por escenario.

A diferencia del pipeline de 2024, **no se usa un maestro de Segmentación/Nodo** -- la
clasificación se deriva directamente de los datos (escenario = primer paso después de "Menu
principal"; resultado = combinación de `paso_ce`, `tipo_desconexion` y `ultima_opcion`).

**Entradas:** `df_pasos_<periodo>.parquet`, `df_traza_<periodo>.parquet`

**Salidas:**
- `df_resumen_<periodo>.parquet` — una fila por conversación: escenario, resultado_final,
  n_pasos, tiempo navegación/técnico, banderas de webservices, recontactos. Insumo del tablero.
- `df_aristas_<periodo>.parquet` — una fila por paso con `op_text` -> `op_text_final`
  (`shift(-1)` dentro de cada conversación). Insumo directo del notebook de grafos.

In [ ]:
import warnings
from pathlib import Path
import re

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

#### Parámetros — deben coincidir con los usados en los notebooks anteriores

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
STAGE_DIR = DATA_DIR / "01_staging"
SEG_DIR = DATA_DIR / "02_segmentacion"
SEG_DIR.mkdir(parents=True, exist_ok=True)

IN_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
IN_TRAZA_PATH = STAGE_DIR / f"df_traza_{FECHA_INICIO}_{FECHA_FIN}.parquet"

OUT_RESUMEN_PATH = SEG_DIR / f"df_resumen_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_ARISTAS_PATH = SEG_DIR / f"df_aristas_{FECHA_INICIO}_{FECHA_FIN}.parquet"

for p in (IN_PASOS_PATH, IN_TRAZA_PATH):
    assert p.exists(), f"No encuentro {p} -- corre primero 01_reconstruccion_traza.ipynb"

In [ ]:
df_pasos = pd.read_parquet(IN_PASOS_PATH)
df_traza = pd.read_parquet(IN_TRAZA_PATH)

print(f"df_pasos: {df_pasos.shape}")
print(f"df_traza: {df_traza.shape}")

df_pasos: (818010, 5)
df_traza: (105499, 80)


#### 1. Escenario (rama del menú principal)

Se toma el `op_text` inmediatamente **después** de la primera vez que aparece "Menu
principal" en la traza de cada conversación. Esto se adapta solo si el IVR cambia el texto
de las opciones -- no depende de una lista fija de 6 nombres.

In [ ]:
df_pasos_ord = df_pasos.sort_values(["id_conversacion", "bloque"]).copy()
df_pasos_ord["es_menu_principal"] = df_pasos_ord["op_text"].str.strip().str.lower() == "menu principal"
df_pasos_ord["siguiente_paso"] = df_pasos_ord.groupby("id_conversacion")["op_text"].shift(-1)

escenarios = (
    df_pasos_ord[df_pasos_ord["es_menu_principal"]]
    .groupby("id_conversacion")["siguiente_paso"].first()
    .rename("escenario")
)

print(f"Escenario detectado en {escenarios.notna().sum():,} de {df_traza['id_conversacion'].nunique():,} conversaciones")
escenarios.value_counts().head(15)

Escenario detectado en 69,692 de 105,499 conversaciones


escenario
Entrega y Otros Tramites           21362
Garantias y devoluciones           18507
Información general                10510
Existencia de producto              7381
Agendar servicio de instalación     4101
Repetir informacion                 2960
Transferencia Alkosto - Tuya        2939
No input                            1387
No match                             545
Name: count, dtype: int64

In [ ]:
# ¿Los cierres "System" tienen una duración razonable, o hay cortes silenciosos muy cortos?
resuelto_por_sistema = df_traza[
    (df_traza["resultado_final"] == "Resuelto en IVR") & (df_traza["tipo_desconexion"] == "System")
]
resuelto_por_finalizacion = df_traza[
    (df_traza["resultado_final"] == "Resuelto en IVR") & (df_traza["ultimo_texto"].str.lower().str.contains("finaliz", na=False))
]

print("Cierre automático (System) -- duracion_ivr:")
print(resuelto_por_sistema["duracion_ivr"].describe())
print("\nFinalización explícita -- duracion_ivr:")
print(resuelto_por_finalizacion["duracion_ivr"].describe())

Cierre automático (System) -- duracion_ivr:
count     12954.000000
mean      84921.644743
std       35060.007358
min        4225.000000
25%       58358.250000
50%       85065.500000
75%      100640.250000
max      352675.000000
Name: duracion_ivr, dtype: float64

Finalización explícita -- duracion_ivr:
count      5178.000000
mean      96817.461954
std       38723.957286
min        5512.000000
25%       68815.750000
50%       91677.500000
75%      118625.250000
max      360126.000000
Name: duracion_ivr, dtype: float64


#### 2. Resultado final de la llamada

**~8-9% de las conversaciones no tienen `traza_opciones`** -- el análisis mostró que el 95%
de esos casos no son abandono ni ruido: son llamadas que **nunca pasaron por el IVR**
(`paso_ce = SI`, `duracion_ivr = 0`, `nombre_ivr = "SinEspecificar"`), enrutadas directo a un
asesor. Se conservan y se clasifican aparte -- no se eliminan ni se mezclan con las que sí
navegaron y terminaron en asesor.

Reglas, en orden de prioridad:
1. Sin traza (`paso_por_ivr == False`):
   - `paso_ce == "SI"` o el asesor colgó (`tipo_desconexion == "Agent"`) ->
     **Transferencia directa (sin pasar por IVR)**
   - si no -> **Colgó antes de cualquier navegación**
2. Con traza:
   - bandera de asesor, texto que lo menciona, o `tipo_desconexion == "Agent"` ->
     **Transferido a asesor**
   - hay traza pero `ultima_opcion` no quedó registrada -> **Sin navegación registrada**
     (anomalía de datos, no se asume resuelto sobre un dato faltante)
   - texto de error de flujo -> **Error técnico**
   - "no input" / "no match" -> **Abandono / sin respuesta**
   - finalización explícita del flujo -> **Resuelto en IVR**
   - `tipo_desconexion == "System"` (nodos informativos terminales que cuelgan sin pasar por
     un mensaje de finalización genérico, ej. "Existencia de producto") -> **Resuelto en IVR**
     -- validado contra `duracion_ivr`: mediana similar a la finalización explícita.
   - `tipo_desconexion == "External"` -> **Cliente colgó en navegación**
   - lo que no cae en ninguna regla -> **Otro / sin clasificar** (debería quedar ~vacío).

In [ ]:
def parsear_paso_simple(campo):
    """De 'codigo;texto;tiempo' (con o sin '|' inicial) devuelve (codigo, texto)."""
    if pd.isna(campo):
        return None, None
    partes = str(campo).strip("|").split(";")
    codigo = partes[0].strip() if len(partes) >= 1 else None
    texto = partes[1].strip() if len(partes) >= 2 else None
    return codigo, texto

codigos_textos = df_traza["ultima_opcion"].apply(parsear_paso_simple)
df_traza["ultimo_codigo"] = codigos_textos.apply(lambda t: t[0])
df_traza["ultimo_texto"] = codigos_textos.apply(lambda t: t[1])

# ¿La conversación llegó a navegar algo del IVR? (traza_opciones no nulo)
df_traza["paso_por_ivr"] = df_traza["traza_opciones"].notna()

patron_asesor = re.compile(r"paso[_\s]*agente|desborde|transferencia", re.IGNORECASE)

def clasificar_resultado(row):
    # -- Sin traza: la llamada nunca navegó el IVR --
    if not row["paso_por_ivr"]:
        if row["paso_ce"] == "SI" or row["tipo_desconexion"] == "Agent":
            return "Transferencia directa (sin pasar por IVR)"
        return "Colgó antes de cualquier navegación"

    tiene_ultima_opcion = isinstance(row["ultimo_texto"], str) and row["ultimo_texto"] != ""
    texto = row["ultimo_texto"] if tiene_ultima_opcion else ""
    texto_low = texto.lower()

    # 1. Transferido a asesor: bandera explícita, texto que lo menciona, o el asesor colgó
    if row["paso_ce"] == "SI" or patron_asesor.search(texto) or row["tipo_desconexion"] == "Agent":
        return "Transferido a asesor"

    # 2. Hay traza pero no quedó registrado el último paso -- anomalía de datos,
    #    no asumir "resuelto" sobre un dato faltante
    if not tiene_ultima_opcion:
        return "Sin navegación registrada"

    # 3. Error de flujo (antes que 'finaliz' genérico, porque el texto de error también lo contiene)
    if "error en el flujo" in texto_low:
        return "Error técnico"

    # 4. Abandono explícito: el cliente no marcó nada o no se le entendió
    if "no input" in texto_low or "no match" in texto_low:
        return "Abandono / sin respuesta"

    # 5. Finalización explícita del flujo (ej. 'Finalización por fin de flujo')
    if "finaliz" in texto_low:
        return "Resuelto en IVR"

    # 6. El sistema cerró la llamada él mismo (no el cliente, no un asesor) -- son nodos
    #    informativos terminales (ej. 'Existencia de producto', 'Fuera de horario default')
    #    que entregan la respuesta y cuelgan sin pasar por un nodo genérico de finalización.
    #    Validado contra duracion_ivr: mediana similar a la finalización explícita.
    if row["tipo_desconexion"] == "System":
        return "Resuelto en IVR"

    # 7. El cliente colgó a mitad de la navegación
    if row["tipo_desconexion"] == "External":
        return "Cliente colgó en navegación"

    return "Otro / sin clasificar"

df_traza["resultado_final"] = df_traza.apply(clasificar_resultado, axis=1)

resumen_resultado = (df_traza["resultado_final"].value_counts(normalize=True) * 100).round(1)
print(resumen_resultado)
print()
print(f"% de llamadas que NUNCA pasaron por el IVR: {(~df_traza['paso_por_ivr']).mean()*100:.1f}%")

resultado_final
Transferido a asesor                         57.2
Resuelto en IVR                              13.8
Cliente colgó en navegación                  13.2
Transferencia directa (sin pasar por IVR)     7.4
Abandono / sin respuesta                      4.3
Error técnico                                 3.6
Colgó antes de cualquier navegación           0.5
Name: proportion, dtype: float64

% de llamadas que NUNCA pasaron por el IVR: 7.9%


In [ ]:
# 3. Longitud de la traza (número de pasos)
n_pasos = df_pasos.groupby("id_conversacion").size().rename("n_pasos")
n_pasos.describe()

count    97151.000000
mean         8.419985
std          5.487240
min          1.000000
25%          5.000000
50%          7.000000
75%         11.000000
max         44.000000
Name: n_pasos, dtype: float64

In [ ]:
# 4. Recontactos (rellamadas)
df_recontactos = df_traza[["id_conversacion", "ani", "fecha_hora_ingreso", "fecha_hora_fin"]].copy()
df_recontactos["fecha_hora_ingreso"] = pd.to_datetime(df_recontactos["fecha_hora_ingreso"])
df_recontactos["fecha_hora_fin"] = pd.to_datetime(df_recontactos["fecha_hora_fin"])
df_recontactos = df_recontactos.sort_values(["ani", "fecha_hora_ingreso"])

df_recontactos["n_contactos_ani"] = df_recontactos.groupby("ani")["ani"].transform("size")
df_recontactos["numero_llamada"] = df_recontactos.groupby("ani").cumcount() + 1
df_recontactos["horas_desde_contacto_anterior"] = (
    df_recontactos["fecha_hora_ingreso"] - df_recontactos.groupby("ani")["fecha_hora_fin"].shift(1)
).dt.total_seconds() / 3600
df_recontactos["unico_contacto"] = df_recontactos["n_contactos_ani"] <= 1

df_recontactos = df_recontactos[[
    "id_conversacion", "n_contactos_ani", "numero_llamada",
    "horas_desde_contacto_anterior", "unico_contacto",
]]

print(f"Conversaciones con más de un contacto: {(~df_recontactos['unico_contacto']).sum():,} de {len(df_recontactos):,}")

Conversaciones con más de un contacto: 82,047 de 105,499


In [ ]:
df_recontactos.head(5)

,id_conversacion,n_contactos_ani,numero_llamada,horas_desde_contacto_anterior,unico_contacto
50311,ce30dc92-cb49-4aed-9dd6-6c7575c4940c,1,1,NaN,True
71118,68565fcf-9f52-43c2-945e-8349d3870199,4,1,NaN,False
67663,4f3e3b59-d698-4965-b1ff-2e51f9f01a53,4,2,4.552831,False
78667,b744bf4b-695b-48f2-ace1-8d4aee15d768,4,3,73.057783,False
84533,609df35c-16fc-486c-b116-d5c91be52113,4,4,18.663950,False


In [ ]:
# 5. Consolidar `df_resumen`
df_resumen = (
    df_traza
    .merge(escenarios, on="id_conversacion", how="left")
    .merge(n_pasos, on="id_conversacion", how="left")
    .merge(df_recontactos, on="id_conversacion", how="left")
)

df_resumen["escenario"] = df_resumen["escenario"].fillna("No llegó al menú principal")
df_resumen["n_pasos"] = df_resumen["n_pasos"].fillna(0).astype(int)

print(f"df_resumen: {df_resumen.shape}")
df_resumen.head(3)

df_resumen: (105499, 90)


,id_conversacion,ani,dnis,nombre_ivr,fecha_hora_ingreso,fecha_hora_fin,duracion_ivr,duracion_navegacion,duracion_transaccional,duracion_paso_ce,duracion_desborde,traza_opciones,ultima_opcion,paso_ce,tipo_desconexion,fecha_inicio,fecha_fin,traza_1,traza_2,traza_3,traza_4,traza_5,traza_6,traza_7,traza_8,traza_9,traza_10,traza_11,traza_12,traza_13,traza_14,traza_15,traza_16,traza_17,traza_18,traza_19,traza_20,traza_21,traza_22,traza_23,traza_24,traza_25,traza_26,traza_27,traza_28,traza_29,traza_30,traza_31,traza_32,traza_33,traza_34,traza_35,traza_36,traza_37,traza_38,traza_39,traza_40,traza_41,traza_42,traza_43,traza_44,documento_ingresado,usuario_identificado_ani,ws_actualiza_habeas_data,ws_consulta_habeas_data,ws_check_aftersales_cases,ws_get_client_by_ani,autorizacion_tramite,numero_guia_transportadora,destino_transferencia,numero_destino_transferencia,marca_producto,producto_texto_libre,verificacion_producto_ws,verificacion_producto_flag,numero_pedido_ingresado,ws_resultado_verificacion_1,ws_resultado_verificacion_2,ws_resultado_verificacion_3,ws_resultado_verificacion_frecuente,ultimo_codigo,ultimo_texto,paso_por_ivr,resultado_final,escenario,n_pasos,n_contactos_ani,numero_llamada,horas_desde_contacto_anterior,unico_contacto
0,891f883b-e8c0-4a43-91a8-00f93459a745,+573212202672,+576014073033,"ALK_IVR_PRINCIPAL,Default In-Queue Flow Alkost...",2026-06-01 18:51:32.530,2026-06-01 19:03:54.720,185663,185095.0,1496.0,284.0,0.0,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,1003;Finalización por fin de flujo;31,SI,System,2026-06-01,2026-06-01,Inicio IVR,Habeas data positivo,Menu principal,Garantias y devoluciones,Repetir informacion,Repeat,Iniciar_Tu_Garantia,Igual_O_Menor_30_Dias,Producto_Deteriorado,Gran_Tamano,Paso agente garantias,Bienvenida encuesta SAC,Primera pregunta SAC,Segunda pregunta SAC,Pasa a buzon = NO,Finalización por fin de flujo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1023941473,NO,FAILURE,FAILURE,OK,NOK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1003,Finalización por fin de flujo,True,Transferido a asesor,Garantias y devoluciones,16,1,1,NaN,True
1,a474401c-9cf1-491b-a677-0ac154ac6141,+573156554180,+576014073033,ALK_IVR_PRINCIPAL,2026-06-01 15:31:24.227,2026-06-01 15:32:22.083,57833,57301.0,1301.0,266.0,0.0,|0;Inicio IVR ;0|999;No input;14897,999;No input;14897,NO,External,2026-06-01,2026-06-01,Inicio IVR,No input,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1007469531,NO,NaN,FAILURE,NaN,NOK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,999,No input,True,Abandono / sin respuesta,No llegó al menú principal,2,1,1,NaN,True
2,352a5eb7-1bfd-484f-820e-12503f2f8c56,+573146826588,+576014073033,ALK_IVR_PRINCIPAL,2026-06-01 12:23:54.010,2026-06-01 12:34:28.390,36453,34771.0,1682.0,NaN,0.0,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,9;Transferencia Alkosto - Tuya;14753,NO,System,2026-06-01,2026-06-01,Inicio IVR,Usuario_Identificado_Con_Ani,Solicitud_Para_mi,Menu principal,Transferencia Alkosto - Tuya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,OK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,Transferencia Alkosto - Tuya,True,Transferido a asesor,Transferencia Alkosto - Tuya,5,3,2,0.037013,False


In [ ]:
# Chequeo cruzado rápido: escenario x resultado_final -- para calibrar antes de exportar
pd.crosstab(df_resumen["escenario"], df_resumen["resultado_final"])

resultado_final,Abandono / sin respuesta,Cliente colgó en navegación,Colgó antes de cualquier navegación,Error técnico,Resuelto en IVR,Transferencia directa (sin pasar por IVR),Transferido a asesor
escenario,,,,,,,
Agendar servicio de instalación,0,0,0,0,238,0,3863
Entrega y Otros Tramites,628,1965,0,2027,2223,0,14519
Existencia de producto,0,141,0,564,6676,0,0
Garantias y devoluciones,292,1607,0,265,2233,0,14110
Información general,47,453,0,10,1341,0,8659
No input,241,64,0,20,208,0,854
No llegó al menú principal,3216,9443,555,639,761,7793,13400
No match,51,28,0,22,102,0,342
Repetir informacion,40,276,0,89,519,0,2036


#### 6. Tabla de aristas (origen → destino) para el grafo

`op_text_final` = el siguiente paso dentro de la misma conversación (`shift(-1)`); la última
fila de cada conversación se marca como `"final"`. Esto reemplaza la función
`transformar_dataframe` de `01_Alk_final.ipynb` (2024).

In [ ]:
df_aristas = df_pasos.sort_values(["id_conversacion", "bloque"]).copy()
df_aristas["op_text_final"] = df_aristas.groupby("id_conversacion")["op_text"].shift(-1)

ultimas_filas = df_aristas.groupby("id_conversacion").tail(1).index
df_aristas.loc[ultimas_filas, "op_text_final"] = "final"

print(f"df_aristas: {df_aristas.shape}")
df_aristas.head(8)

df_aristas: (818010, 6)


,id_conversacion,bloque,op_num,op_text,op_tiempo,op_text_final
83050,0000b8f3-39e8-4388-aa08-23c0031a9d9f,0,0,Inicio IVR,0,Habeas data negativo
83051,0000b8f3-39e8-4388-aa08-23c0031a9d9f,1,4,Habeas data negativo,47589,Menu principal
83052,0000b8f3-39e8-4388-aa08-23c0031a9d9f,2,17,Menu principal,604,Agendar servicio de instalación
83053,0000b8f3-39e8-4388-aa08-23c0031a9d9f,3,11,Agendar servicio de instalación,18954,Paso agente instalaciones
83054,0000b8f3-39e8-4388-aa08-23c0031a9d9f,4,206,Paso agente instalaciones,124,Bienvenida encuesta SAC
83055,0000b8f3-39e8-4388-aa08-23c0031a9d9f,5,900,Bienvenida encuesta SAC,600812,Primera pregunta SAC
83056,0000b8f3-39e8-4388-aa08-23c0031a9d9f,6,901,Primera pregunta SAC,26,Segunda pregunta SAC
83057,0000b8f3-39e8-4388-aa08-23c0031a9d9f,7,902,Segunda pregunta SAC,14757,Pasa a buzon = NO


#### 7. Exportar

In [ ]:
df_resumen.to_parquet(OUT_RESUMEN_PATH, index=False)
df_aristas.to_parquet(OUT_ARISTAS_PATH, index=False)

print("Guardado:")
print(f"  {OUT_RESUMEN_PATH} -> {df_resumen.shape}  (insumo del tablero Power BI)")
print(f"  {OUT_ARISTAS_PATH} -> {df_aristas.shape}  (insumo del grafo, filtrar por escenario aquí)")

# Muestras en Excel para revisión manual
df_resumen.head(500).to_excel(SEG_DIR / f"df_resumen_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx", index=False)
df_aristas.head(500).to_excel(SEG_DIR / f"df_aristas_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx", index=False)

Guardado:
  data\02_segmentacion\df_resumen_2026-06-01_2026-06-30.parquet -> (105499, 90)  (insumo del tablero Power BI)
  data\02_segmentacion\df_aristas_2026-06-01_2026-06-30.parquet -> (818010, 6)  (insumo del grafo, filtrar por escenario aquí)
